# MetroRide with Spark ML: predicting driver arrival time

When a driver **accepts** a trip, MetroRide wants to tell the passenger how long the driver will take to reach them. This notebook trains a regression model for that wait with **Spark ML** (`pyspark.ml`).

**Target (label):** `driver_arrival_wait_minutes` = minutes from `requested_at` to the driver's `arrived_at_pickup` app event. This is the same measure as in `silver.trip_context` (phase 2).

**Prediction moment:** right after acceptance. A feature is allowed only if we know its value at that moment.

| Step | Time | What you do |
|---|---|---|
| 0 | 2 min | Start Spark |
| 1 | 4 min | Build one training row per completed trip, with its label |
| 2 | 2 min | Look at the label |
| 3 | 3 min | Split by time and score a baseline |
| 4 | 5 min | Build a Spark ML pipeline and compare feature sets |
| 5 | 4 min | Find where the model is wrong and add rain |
| | | Summary and *Try it* exercises |

**Data:** the bronze CSVs in `../phases/*/bronze/`. June–September have app events, so they have a label. Only September has rainfall.

**Requirements:** Java 17 or newer, and `pip install pyspark "pandas<3"` (or `uv add pyspark "pandas<3"`). Run the notebook from the `ml/` folder of the case.

> The data is **synthetic** and small (about 500 labelled trips). The goal is to learn the ML workflow (label, features, time split, baseline, pipeline, evaluation), not to find real facts about Bangkok traffic.

## 0. Start Spark

The same session settings as in the PySpark tutorial: Bangkok time zone and few shuffle partitions for small data.

In [1]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

CASE_DIR = Path.cwd().parent                      # .../case/ride-hailing
PHASES = CASE_DIR / "phases"
assert PHASES.exists(), "Run this notebook from the ml/ folder of the ride-hailing case"

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("metroride-spark-ml")
    .config("spark.sql.session.timeZone", "Asia/Bangkok")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

Spark 4.2.0


## 1. Build the training table

An ML model needs **one row per example**, with feature columns and one label column. Our example is one **completed trip**.

1. Read every trip file and every driver-event file as text, then parse the timestamps.
2. Keep each `arrived_at_pickup` event once (the app stream can deliver an event twice).
3. **Inner join** trips with arrivals. This drops cancelled trips (no arrival, so no label) and April–May trips (no app events yet).
4. **Left join** the September rainfall by date and time slot. June–August get `NULL`.

In the full lakehouse this table is `silver.trip_context`. Here we rebuild a small version straight from the CSVs, so the notebook does not depend on the earlier tutorials.

In [2]:
TS = "yyyy-MM-dd'T'HH:mm:ssXXX"          # e.g. 2026-06-01T08:03:00+07:00


def read_text_csv(pattern, columns):
    # Read all matching CSVs with every column as text, like bronze.
    schema = T.StructType([T.StructField(col, T.StringType()) for col in columns])
    paths = [str(p) for p in sorted(PHASES.glob(f"*/bronze/{pattern}"))]
    return spark.read.option("header", True).schema(schema).csv(paths)


def ts(column):
    return F.try_to_timestamp(F.col(column), F.lit(TS))


trips = (read_text_csv("trips_*.csv", [
            "trip_id", "driver_id", "service_zone", "requested_at", "accepted_at",
            "picked_up_at", "completed_at", "cancelled_at", "status", "fare_thb"])
         .filter(F.col("status") == "completed")
         .select("trip_id", "driver_id", "service_zone",
                 ts("requested_at").alias("requested_at"),
                 ts("accepted_at").alias("accepted_at"),
                 ts("picked_up_at").alias("picked_up_at")))

arrivals = (read_text_csv("driver_events_*.csv", [
               "event_id", "trip_id", "driver_id", "event_type", "event_at", "latitude", "longitude"])
            .filter(F.col("event_type") == "arrived_at_pickup")
            .dropDuplicates(["event_id"])
            .select("trip_id", ts("event_at").alias("driver_arrived_at")))

weather = (read_text_csv("weather_*.csv", ["observed_at", "city", "time_slot", "rain_mm", "source_page"])
           .select(F.to_date(ts("observed_at")).alias("service_date"),
                   "time_slot",
                   F.col("rain_mm").try_cast("double").alias("rain_mm")))

print("completed trips:", trips.count(), "| arrival events:", arrivals.count(), "| weather rows:", weather.count())

completed trips: 752 | arrival events: 501 | weather rows: 60


Now join and compute the label and the features. `unix_timestamp()` gives seconds, so we divide by 60 for minutes.

| Column | Kind | Known at acceptance? |
|---|---|---|
| `service_zone`, `time_slot`, `weekday` | categorical feature | yes |
| `accept_delay_minutes` = `accepted_at - requested_at` | numeric feature | yes |
| `is_rainy` (1.0 if `rain_mm >= 1.0`) | numeric feature, September only | yes, from the weather observation of that slot |
| `passenger_pickup_wait_minutes` = `picked_up_at - requested_at` | not a feature | **no**: the passenger is picked up *after* the driver arrives |
| `driver_arrival_wait_minutes` | **label** | no: this is what we predict |

`month` is not a feature either: we only use it to split the data. We keep `passenger_pickup_wait_minutes` in the table for a *Try it* exercise on leakage.

In [3]:
def minutes_between(start, end):
    return (F.unix_timestamp(end) - F.unix_timestamp(start)) / 60


data = (trips
    .join(arrivals, "trip_id", "inner")
    .withColumn("service_date", F.to_date("requested_at"))
    .withColumn("time_slot", F.when(F.hour("requested_at") == 8, "morning").otherwise("evening"))
    .join(weather, ["service_date", "time_slot"], "left")
    .select(
        "trip_id", "service_date",
        F.month("service_date").alias("month"),
        "service_zone", "time_slot",
        F.date_format("requested_at", "EEE").alias("weekday"),
        minutes_between("requested_at", "accepted_at").alias("accept_delay_minutes"),
        F.when(F.col("rain_mm") >= 1.0, 1.0).when(F.col("rain_mm").isNotNull(), 0.0).alias("is_rainy"),
        "rain_mm",
        minutes_between("requested_at", "picked_up_at").alias("passenger_pickup_wait_minutes"),
        minutes_between("requested_at", "driver_arrived_at").alias("driver_arrival_wait_minutes"),
    )
    .cache())

assert data.count() == data.select("trip_id").distinct().count(), "a join duplicated trips"
print(data.count(), "labelled trips")
data.orderBy("trip_id").show(5)

501 labelled trips


+---------------+------------+-----+------------+---------+-------+--------------------+--------+-------+-----------------------------+---------------------------+
|        trip_id|service_date|month|service_zone|time_slot|weekday|accept_delay_minutes|is_rainy|rain_mm|passenger_pickup_wait_minutes|driver_arrival_wait_minutes|
+---------------+------------+-----+------------+---------+-------+--------------------+--------+-------+-----------------------------+---------------------------+
|BKK-20260601-E1|  2026-06-01|    6|         Ari|  evening|    Mon|                 1.0|    NULL|   NULL|                          8.0|                        7.0|
|BKK-20260601-E2|  2026-06-01|    6|       Silom|  evening|    Mon|                 2.0|    NULL|   NULL|                         11.0|                        9.0|
|BKK-20260601-M1|  2026-06-01|    6|        Asok|  morning|    Mon|                 2.0|    NULL|   NULL|                          8.0|                        6.0|
|BKK-20260601-M2

## 2. Look at the label

Always look at the label before modelling. The `mean` is our first "model", and the standard deviation tells us how much there is to explain.

In [4]:
data.groupBy("month").agg(
    F.count("*").alias("trips"),
    F.round(F.avg("driver_arrival_wait_minutes"), 2).alias("avg_wait"),
    F.round(F.stddev("driver_arrival_wait_minutes"), 2).alias("std_wait"),
    F.count("is_rainy").alias("trips_with_weather"),
).orderBy("month").show()

+-----+-----+--------+--------+------------------+
|month|trips|avg_wait|std_wait|trips_with_weather|
+-----+-----+--------+--------+------------------+
|    6|  124|    8.06|    2.43|                 0|
|    7|  127|    8.03|    2.44|                 0|
|    8|  126|    7.95|    2.39|                 0|
|    9|  124|    8.03|    2.46|               124|
+-----+-----+--------+--------+------------------+



Does the wait differ by zone or time slot? These look like obvious features. Keep the answer in mind for Step 4.

In [5]:
for col in ["service_zone", "time_slot"]:
    (data.groupBy(col)
         .agg(F.count("*").alias("trips"),
              F.round(F.avg("driver_arrival_wait_minutes"), 2).alias("avg_wait"))
         .orderBy(col).show())

+------------+-----+--------+
|service_zone|trips|avg_wait|
+------------+-----+--------+
|         Ari|  123|    7.61|
|        Asok|  124|    8.12|
|        Siam|  128|     8.4|
|       Silom|  126|    7.94|
+------------+-----+--------+



+---------+-----+--------+
|time_slot|trips|avg_wait|
+---------+-----+--------+
|  evening|  253|    8.07|
|  morning|  248|    7.97|
+---------+-----+--------+



## 3. Split by time, then score a baseline

**Never split time-ordered data at random.** A random split lets the model learn from September trips and then be tested on other September trips from the same days. In production we always predict the *future* from the *past*, so we test that way too:

* **train:** June–August
* **test:** September

**Metric:** mean absolute error (MAE), in minutes: "on average, the prediction is off by this many minutes". We also show RMSE (punishes big misses more) and R² (share of variance explained; 0 means no better than the mean).

**Baseline:** predict the training mean for every trip. A model is useful only if it beats this.

In [6]:
from pyspark.ml.evaluation import RegressionEvaluator

LABEL = "driver_arrival_wait_minutes"
train = data.filter(F.col("month") <= 8)
test = data.filter(F.col("month") == 9)
print("train:", train.count(), "trips | test:", test.count(), "trips")

results = []


def score(name, predictions):
    # Add one row of test metrics to `results` and print it.
    row = {"model": name}
    for metric in ["mae", "rmse", "r2"]:
        evaluator = RegressionEvaluator(labelCol=LABEL, predictionCol="prediction", metricName=metric)
        row[metric] = round(evaluator.evaluate(predictions), 3)
    results.append(row)
    print(row)


train_mean = train.agg(F.avg(LABEL)).first()[0]
score("baseline: training mean", test.withColumn("prediction", F.lit(train_mean)))

train: 377 trips | test: 124 trips


{'model': 'baseline: training mean', 'mae': 1.971, 'rmse': 2.449, 'r2': -0.0}


## 4. A Spark ML pipeline

Spark ML models read **one vector column** of features. A `Pipeline` chains the steps that build it, then fits the model:

| Stage | Does |
|---|---|
| `StringIndexer` | text category → number, most frequent first (`Siam` → 0.0, `Asok` → 1.0, …) |
| `OneHotEncoder` | number → 0/1 vector, so the model does not treat `Ari` (3.0) as "3 × Asok" |
| `VectorAssembler` | all feature columns → one `features` vector |
| `LinearRegression` | learns one weight per feature plus an intercept |

`pipeline.fit(train)` learns everything (category indexes and weights) from the **training data only**. `model.transform(test)` applies the learned steps to the test data and adds a `prediction` column.

`fit_and_score()` builds the pipeline for a list of features, so we can compare feature sets with one line each.

In [7]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression


def fit_and_score(name, categorical, numeric, train_df=train, test_df=test):
    # Fit a linear regression pipeline on train_df, score it on test_df, return the fitted model.
    stages = []
    if categorical:
        stages += [
            StringIndexer(inputCols=categorical, outputCols=[c + "_idx" for c in categorical], handleInvalid="keep"),
            OneHotEncoder(inputCols=[c + "_idx" for c in categorical], outputCols=[c + "_ohe" for c in categorical]),
        ]
    stages += [
        VectorAssembler(inputCols=numeric + [c + "_ohe" for c in categorical], outputCol="features"),
        LinearRegression(featuresCol="features", labelCol=LABEL),
    ]
    model = Pipeline(stages=stages).fit(train_df)
    score(name, model.transform(test_df))
    return model

**Model 1: the obvious features.** Zone, time slot and weekday.

In [8]:
m1 = fit_and_score("zone + slot + weekday", ["service_zone", "time_slot", "weekday"], [])

{'model': 'zone + slot + weekday', 'mae': 2.029, 'rmse': 2.483, 'r2': -0.028}


MAE is about the same as the baseline, or worse. Step 2 already hinted why: the average wait barely changes by zone or slot, so these columns carry **no signal**. Adding features that carry no signal does not help; it only adds noise to the weights.

**Model 2: add `accept_delay_minutes`.** A driver who took longer to accept starts later, so the passenger waits longer. We know this number at the prediction moment.

In [9]:
m2 = fit_and_score("+ accept delay", ["service_zone", "time_slot", "weekday"], ["accept_delay_minutes"])

{'model': '+ accept delay', 'mae': 1.901, 'rmse': 2.312, 'r2': 0.109}


A small but real improvement. Look inside the pipeline: the last stage is the fitted `LinearRegressionModel`. The first weight belongs to `accept_delay_minutes`, the first column we gave `VectorAssembler`.

In [10]:
lr2 = m2.stages[-1]
print(f"intercept: {lr2.intercept:.2f} min | accept delay weight: {lr2.coefficients[0]:.2f} min per minute")

intercept: 5.68 min | accept delay weight: 1.15 min per minute


## 5. Where is the model wrong?

Before adding features at random, look at the **errors**. Group September's residuals (`label - prediction`) by weather. September is the only month where we know the weather.

In [11]:
(m2.transform(test)
   .withColumn("residual", F.col(LABEL) - F.col("prediction"))
   .groupBy("is_rainy")
   .agg(F.count("*").alias("trips"),
        F.round(F.avg("residual"), 2).alias("avg_residual"),
        F.round(F.avg(F.abs("residual")), 2).alias("mae"))
   .orderBy("is_rainy").show())

+--------+-----+------------+----+
|is_rainy|trips|avg_residual| mae|
+--------+-----+------------+----+
|     0.0|   92|       -1.04|1.51|
|     1.0|   32|        3.04|3.04|
+--------+-----+------------+----+



On rainy slots (`is_rainy = 1.0`) the model is about **3 minutes too optimistic**; on dry slots it is slightly too pessimistic. Rain is the missing feature.

But June–August have **no weather data**, so we cannot train on them with `is_rainy`. We build a second, smaller experiment on **September only**, still split by time:

* **train:** 1–20 September
* **test:** 21–30 September

We compare the same model with and without `is_rainy` on exactly the same split.

In [12]:
sept = data.filter(F.col("month") == 9)
sept_train = sept.filter(F.dayofmonth("service_date") <= 20)
sept_test = sept.filter(F.dayofmonth("service_date") > 20)
print("Sept train:", sept_train.count(), "| Sept test:", sept_test.count(),
      "| rainy test trips:", sept_test.filter("is_rainy = 1.0").count())

sept_mean = sept_train.agg(F.avg(LABEL)).first()[0]
score("Sept: baseline", sept_test.withColumn("prediction", F.lit(sept_mean)))
m3 = fit_and_score("Sept: accept delay", [], ["accept_delay_minutes"], sept_train, sept_test)
m4 = fit_and_score("Sept: accept delay + rain", [], ["accept_delay_minutes", "is_rainy"], sept_train, sept_test)

Sept train: 82 | Sept test: 42 | rainy test trips: 11


{'model': 'Sept: baseline', 'mae': 1.981, 'rmse': 2.395, 'r2': -0.0}


{'model': 'Sept: accept delay', 'mae': 1.787, 'rmse': 2.207, 'r2': 0.151}


{'model': 'Sept: accept delay + rain', 'mae': 1.242, 'rmse': 1.438, 'r2': 0.64}


The rain model reads like a sentence:

In [13]:
lr4 = m4.stages[-1]
print(f"wait ≈ {lr4.intercept:.2f} + {lr4.coefficients[0]:.2f} × accept delay + {lr4.coefficients[1]:.2f} if rainy   (minutes)")

wait ≈ 5.17 + 0.88 × accept delay + 4.23 if rainy   (minutes)


### All results

In [14]:
spark.createDataFrame(results).select("model", "mae", "rmse", "r2").show(truncate=False)

+-------------------------+-----+-----+------+
|model                    |mae  |rmse |r2    |
+-------------------------+-----+-----+------+
|baseline: training mean  |1.971|2.449|-0.0  |
|zone + slot + weekday    |2.029|2.483|-0.028|
|+ accept delay           |1.901|2.312|0.109 |
|Sept: baseline           |1.981|2.395|-0.0  |
|Sept: accept delay       |1.787|2.207|0.151 |
|Sept: accept delay + rain|1.242|1.438|0.64  |
+-------------------------+-----+-----+------+



## Summary

* **One row per example**, a **label**, and features that are **known at the prediction moment**.
* **Split by time**: train on the past, test on the future.
* **Always beat a baseline.** Model 1 shows that "obvious" features can carry no signal.
* A Spark ML **`Pipeline`** (`StringIndexer` → `OneHotEncoder` → `VectorAssembler` → `LinearRegression`) learns from the training data only and is applied to new data with `transform()`.
* **Look at the errors** to find the next feature: grouping residuals by weather pointed to rain.
* The rows from Steps 3–4 (tested on all of September) and from Step 5 (tested on 21–30 September) use **different test sets**: compare MAE only inside each group.

**Caveats.** The data is synthetic: the fixture generator makes arrival wait depend on acceptance delay, a repeating pseudo-random pattern, and +4 minutes in rainy slots, and nothing else. The model rediscovers those rules. With real data, the rain effect is an **association**: rain also brings more demand and more traffic, so it does not prove that rain alone causes the delay.

## Try it

1. **Leakage.** Add the passenger pickup wait (`picked_up_at - requested_at`) as a feature to Model 2. What happens to MAE, and why must this model never be deployed?
2. **Another algorithm.** Replace `LinearRegression` with `RandomForestRegressor` (`pyspark.ml.regression`) in Model 2. Is it better on this data?
3. **Classification.** Predict whether the driver arrives within 8 minutes (`label = 1.0 if wait <= 8`). Use `LogisticRegression` and `BinaryClassificationEvaluator` (area under ROC).

---
## Solutions

Try the exercises yourself before reading these.

**Exercise 1: leakage.** The MAE drops to **0.00** and R² to 1.0: a "perfect" model, which is the classic sign of leakage. The passenger pickup wait *contains* the driver arrival wait, because the passenger is picked up a little after the driver arrives. In this synthetic data that gap even equals the acceptance delay, so the model can compute the label exactly; real data would give a suspiciously small error rather than zero. At the prediction moment the pickup has not happened yet, so the deployed model would have nothing to put in this column.

In [15]:
fit_and_score("LEAKY: + passenger pickup wait", ["service_zone", "time_slot", "weekday"],
              ["accept_delay_minutes", "passenger_pickup_wait_minutes"])
_ = results.pop()          # keep the leaky model out of the results table

{'model': 'LEAKY: + passenger pickup wait', 'mae': 0.0, 'rmse': 0.0, 'r2': 1.0}


**Exercise 2: random forest.** The pipeline stays the same; only the last stage changes. The forest is only a little better than linear regression (compare with `+ accept delay`): a more flexible algorithm cannot find a signal that the features do not contain. Rain helped far more than changing the algorithm.

In [16]:
from pyspark.ml.regression import RandomForestRegressor

cat = ["service_zone", "time_slot", "weekday"]
rf_pipeline = Pipeline(stages=[
    StringIndexer(inputCols=cat, outputCols=[c + "_idx" for c in cat], handleInvalid="keep"),
    OneHotEncoder(inputCols=[c + "_idx" for c in cat], outputCols=[c + "_ohe" for c in cat]),
    VectorAssembler(inputCols=["accept_delay_minutes"] + [c + "_ohe" for c in cat], outputCol="features"),
    RandomForestRegressor(featuresCol="features", labelCol=LABEL, numTrees=50, maxDepth=4, seed=42),
])
score("random forest: + accept delay", rf_pipeline.fit(train).transform(test))
_ = results.pop()

{'model': 'random forest: + accept delay', 'mae': 1.871, 'rmse': 2.295, 'r2': 0.122}


**Exercise 3: classification.** Same features, a 0/1 label and a classifier. The evaluator reads the `rawPrediction` column that `LogisticRegression` adds. An area under ROC of 0.5 is random guessing.

In [17]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

with_target = lambda df: df.withColumn("within_8", (F.col(LABEL) <= 8).cast("double"))
clf = Pipeline(stages=[
    VectorAssembler(inputCols=["accept_delay_minutes", "is_rainy"], outputCol="features"),
    LogisticRegression(featuresCol="features", labelCol="within_8"),
]).fit(with_target(sept_train))

auc = BinaryClassificationEvaluator(labelCol="within_8", metricName="areaUnderROC").evaluate(
    clf.transform(with_target(sept_test)))
print(f"area under ROC on 21-30 September: {auc:.3f}")

area under ROC on 21-30 September: 0.943


In [18]:
spark.stop()